In [1]:
# --- bootstrap: anchor to the repository root, wherever this notebook was opened from ---
# Notebooks live two levels deep under notebooks/, so the cwd-relative path logic below needs the
# root established first. Keyed on pytest.ini, which is not tied to any folder-naming decision.
import os
import sys
from pathlib import Path

_root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pytest.ini").exists())
os.chdir(_root)
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))
print(f"repo root: {_root}")

# CRSP Sector ETF Price Extraction — Data Collection

**Academic research only — not investment advice.**

This is the market-data counterpart to `01_ravenpack_news_extraction.ipynb`, built in the same silver/gold shape. It pulls daily price/return/volume history for the eleven SPDR sector ETFs (the index-level proxies used throughout this project, not individual stocks) from `crsp.stksecurityinfohist` (security/ticker history) joined to `crsp.dsf_v2` (CRSP CIZ daily stock file) — the schema validated in `Basic_EDA_Analysis.ipynb`.

Two outputs:

1. **Bronze/Silver — raw daily security pull** (`data/raw/crsp_sector_etf_daily_raw_<START>_<END>.csv`): one row per ETF per CRSP trading session, straight off the join, no engineered features.
**Overnight gap (added 2026-08-07):** the pull now also takes `d.dlyopen`, and the gold panel carries `overnight_gap` (prev close → open) and `open_to_close` (open → close). Pre-open news is absorbed in the gap, which is *not* capturable by trading at the open — separating the two is what lets the modelling notebooks distinguish a research finding about predictability from a tradeable one.

2. **Gold — `market_daily_df`** (`data/market_daily_df.csv`): cleaned, deduplicated, mapped to sector asset names, with 1-day and 5-day forward return/label features engineered for the baseline vs. sentiment-augmented classifier comparison.

**Licensing note:** CRSP is also a licensed WRDS dataset. As a precaution, the row-level pull is written only to `data/raw/`, which is `.gitignore`'d — same conservative treatment as the RavenPack silver table. Only the engineered gold table is intended to be committed.

In [2]:
%pip install -q wrds psycopg2-binary pandas numpy pyarrow

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import wrds

warnings.filterwarnings("ignore", category=FutureWarning)
pd.options.display.max_columns = 120

PROJECT_ROOT = Path.cwd()
if (PROJECT_ROOT / "data").is_dir():
    REPO_ROOT = PROJECT_ROOT
    NOTEBOOK_DIR = REPO_ROOT / "data"
elif PROJECT_ROOT.name == "data" and (PROJECT_ROOT.parent / "data").is_dir():
    REPO_ROOT = PROJECT_ROOT.parent
    NOTEBOOK_DIR = PROJECT_ROOT
else:
    raise FileNotFoundError("Run this notebook from the repository root or data/.")
RAW_DIR = NOTEBOOK_DIR / "raw"
RAW_DIR.mkdir(exist_ok=True)

START_DATE = "2015-01-01"
END_DATE = "2026-12-31"

SECTOR_ETF_TO_ASSET = {
    "XLK": "Technology",
    "XLV": "Health_Care",
    "XLF": "Financials",
    "XLC": "Communication_Services",
    "XLY": "Consumer_Discretionary",
    "XLI": "Industrials",
    "XLP": "Consumer_Staples",
    "XLE": "Energy",
    "XLU": "Utilities",
    "XLB": "Materials",
    "XLRE": "Real_Estate",
}
TARGET_TICKERS = tuple(SECTOR_ETF_TO_ASSET.keys())
FORWARD_HORIZONS = [1, 5]

RAW_PRICES_CSV = RAW_DIR / f"crsp_sector_etf_daily_raw_{START_DATE[:4]}_{END_DATE[:4]}.csv"
MARKET_DAILY_CSV = NOTEBOOK_DIR / "market_daily_df.csv"

print(f"Date range: {START_DATE} to {END_DATE}")
print(f"Sector ETF universe: {TARGET_TICKERS}")
print(f"Bronze/Silver output (gitignored, raw/): {RAW_PRICES_CSV}")
print(f"Gold output (committed, engineered):     {MARKET_DAILY_CSV}")

In [4]:
# Connect to WRDS. This may prompt for credentials if no .pgpass file is configured.
db = wrds.Connection()

crsp_tables = db.list_tables(library="crsp")
required_crsp_tables = ["stksecurityinfohist", "dsf_v2"]
missing_crsp_tables = [table for table in required_crsp_tables if table not in crsp_tables]

if missing_crsp_tables:
    raise RuntimeError(f"Missing CRSP tables: {missing_crsp_tables}")

print("WRDS connection ready.")
print("Required CRSP tables found:", required_crsp_tables)

Loading library list...


Done
WRDS connection ready.
Required CRSP tables found: ['stksecurityinfohist', 'dsf_v2']


## 1. Pull raw CRSP daily price/return/volume for the sector ETFs (Bronze/Silver)

`crsp.stksecurityinfohist` gives the ticker-to-permno mapping with valid date ranges; `crsp.dsf_v2` (CRSP CIZ) has the daily return/price/volume. This is the schema that was validated against the broad-market ETF proxies in `Basic_EDA_Analysis.ipynb` ("CRSP CIZ crsp.dsf_v2"), now pointed at the eleven SPDR sector ETFs.

In [5]:
def sql_string_list(values):
    """Return a SQL-safe single-quoted literal list for simple identifier strings."""
    return ", ".join("'" + str(value).replace("'", "''") + "'" for value in values)


ticker_sql = sql_string_list(TARGET_TICKERS)

market_query = f"""
    WITH names AS (
        SELECT DISTINCT
            permno,
            ticker,
            secinfostartdt AS name_start,
            COALESCE(secinfoenddt, DATE '9999-12-31') AS name_end
        FROM crsp.stksecurityinfohist
        WHERE ticker IN ({ticker_sql})
          AND secinfostartdt <= DATE '{END_DATE}'
          AND COALESCE(secinfoenddt, DATE '9999-12-31') >= DATE '{START_DATE}'
    )
    SELECT
        d.dlycaldt AS session_date,
        n.ticker,
        d.permno,
        d.dlyret AS daily_return,
        d.dlyprc AS price,
        d.dlyopen AS open_price,
        d.dlyvol AS volume
    FROM crsp.dsf_v2 d
    INNER JOIN names n
        ON d.permno = n.permno
       AND d.dlycaldt BETWEEN n.name_start AND n.name_end
    WHERE d.dlycaldt BETWEEN DATE '{START_DATE}' AND DATE '{END_DATE}'
    ORDER BY n.ticker, d.dlycaldt
"""

market_daily_raw = db.raw_sql(market_query)

tickers_found = set(market_daily_raw["ticker"].dropna().unique())
missing_tickers = set(TARGET_TICKERS) - tickers_found
if missing_tickers:
    raise RuntimeError(f"Could not retrieve all sector ETF tickers from CRSP: missing {missing_tickers}")

print(f"Raw CRSP rows pulled: {len(market_daily_raw):,}")
print(f"Tickers found: {sorted(tickers_found)}")
display(market_daily_raw.head())

Raw CRSP rows pulled: 29,362
Tickers found: ['XLB', 'XLC', 'XLE', 'XLF', 'XLI', 'XLK', 'XLP', 'XLRE', 'XLU', 'XLV', 'XLY']


,session_date,ticker,permno,daily_return,price,open_price,volume
0,2015-01-02,XLB,86449,0.001441,48.65,48.58,5234797.0
1,2015-01-05,XLB,86449,-0.025488,47.41,48.28,5438843.0
2,2015-01-06,XLB,86449,-0.008859,46.99,47.55,5054469.0
3,2015-01-07,XLB,86449,0.011279,47.52,47.34,3560313.0
4,2015-01-08,XLB,86449,0.02378,48.65,48.0,7640540.0


In [6]:
# Licensed WRDS data — write only to the gitignored raw/ folder, never to a tracked path.
market_daily_raw.to_csv(RAW_PRICES_CSV, index=False)
print(f"Saved bronze/silver (raw CRSP pull) table to: {RAW_PRICES_CSV}")
print("This path is under data/raw/, which is .gitignore'd - do not force-add it.")

## 2. Clean & engineer the daily sector-asset panel (Gold): `market_daily_df`

Dedup, map ticker → sector asset name, and engineer 1-day and 5-day forward return/label features — same logic as the ETF proxy panel in `Basic_EDA_Analysis.ipynb`, applied to the eleven SPDR sector ETFs.

In [7]:
market_daily_df = market_daily_raw.copy()
market_daily_df["session_date"] = pd.to_datetime(market_daily_df["session_date"])
market_daily_df["daily_return"] = pd.to_numeric(market_daily_df["daily_return"], errors="coerce")
market_daily_df["price"] = pd.to_numeric(market_daily_df["price"], errors="coerce").abs()
# CRSP encodes a bid/ask average as a negative price; .abs() is applied to open for the same reason.
market_daily_df["open_price"] = pd.to_numeric(market_daily_df["open_price"], errors="coerce").abs()
market_daily_df["volume"] = pd.to_numeric(market_daily_df["volume"], errors="coerce")
market_daily_df["asset"] = market_daily_df["ticker"].map(SECTOR_ETF_TO_ASSET)

market_daily_df = (
    market_daily_df
    .dropna(subset=["asset", "session_date"])
    .sort_values(["asset", "session_date", "permno"])
)
duplicate_asset_sessions = market_daily_df.duplicated(["asset", "session_date"]).sum()
if duplicate_asset_sessions:
    raise ValueError(f"CRSP extraction has {duplicate_asset_sessions:,} duplicate asset/session rows; resolve security-history overlap before continuing.")
market_daily_df = market_daily_df.reset_index(drop=True)

# Decompose the session into the part a trader cannot capture (the overnight gap, which is where
# pre-open news is absorbed) and the part they can (open -> close). These are raw price ratios rather
# than distribution-adjusted returns, so they will not compound exactly to `daily_return` on
# ex-dividend dates; the discrepancy is a few basis points and is checked in section 3.
market_daily_df["prev_close"] = market_daily_df.groupby("asset")["price"].shift(1)
market_daily_df["overnight_gap"] = market_daily_df["open_price"] / market_daily_df["prev_close"] - 1
market_daily_df["open_to_close"] = market_daily_df["price"] / market_daily_df["open_price"] - 1
market_daily_df["gap_positive"] = np.where(
    market_daily_df["overnight_gap"].notna(),
    (market_daily_df["overnight_gap"] > 0).astype(float),
    np.nan,
)
market_daily_df["open_to_close_positive"] = np.where(
    market_daily_df["open_to_close"].notna(),
    (market_daily_df["open_to_close"] > 0).astype(float),
    np.nan,
)

for horizon in FORWARD_HORIZONS:
    shifted_product = pd.Series(1.0, index=market_daily_df.index)
    for lag in range(1, horizon + 1):
        shifted_product = shifted_product * (1 + market_daily_df.groupby("asset")["daily_return"].shift(-lag))
    fwd_col = f"fwd_{horizon}d_return"
    label_col = f"fwd_{horizon}d_positive"
    market_daily_df[fwd_col] = shifted_product - 1
    market_daily_df[label_col] = np.where(
        market_daily_df[fwd_col].notna(),
        (market_daily_df[fwd_col] > 0).astype(float),
        np.nan,
    )

print(f"Gold panel shape: {market_daily_df.shape}")
print(market_daily_df.groupby(["asset", "ticker"]).agg(
    start=("session_date", "min"),
    end=("session_date", "max"),
    sessions=("session_date", "nunique"),
    nonnull_returns=("daily_return", "count"),
))
display(market_daily_df.head())

Gold panel shape: (29362, 17)
                                   start        end  sessions  nonnull_returns
asset                  ticker                                                 
Communication_Services XLC    2018-06-19 2025-12-31      1895             1894
Consumer_Discretionary XLY    2015-01-02 2025-12-31      2766             2766
Consumer_Staples       XLP    2015-01-02 2025-12-31      2766             2766
Energy                 XLE    2015-01-02 2025-12-31      2766             2766
Financials             XLF    2015-01-02 2025-12-31      2766             2766
Health_Care            XLV    2015-01-02 2025-12-31      2766             2766
Industrials            XLI    2015-01-02 2025-12-31      2766             2766
Materials              XLB    2015-01-02 2025-12-31      2766             2766
Real_Estate            XLRE   2015-10-08 2025-12-31      2573             2572
Technology             XLK    2015-01-02 2025-12-31      2766             2766
Utilities             

,session_date,ticker,permno,daily_return,price,open_price,volume,asset,prev_close,overnight_gap,open_to_close,gap_positive,open_to_close_positive,fwd_1d_return,fwd_1d_positive,fwd_5d_return,fwd_5d_positive
0,2018-06-19,XLC,17940,<NA>,49.96,49.7,16588.0,Communication_Services,<NA>,<NA>,0.005231,NaN,1.0,0.01241,1.0,-0.008563,0.0
1,2018-06-20,XLC,17940,0.01241,50.58,50.45,189989.0,Communication_Services,49.96,0.009808,0.002577,1.0,1.0,-0.006129,0.0,-0.029262,0.0
2,2018-06-21,XLC,17940,-0.006129,50.27,50.77,428740.0,Communication_Services,50.58,0.003756,-0.009848,1.0,0.0,0.004376,1.0,-0.013527,0.0
3,2018-06-22,XLC,17940,0.004376,50.49,50.59,181638.0,Communication_Services,50.27,0.006366,-0.001977,1.0,0.0,-0.020598,0.0,-0.019014,0.0
4,2018-06-25,XLC,17940,-0.020598,49.45,50.23,2509603.0,Communication_Services,50.49,-0.00515,-0.015529,0.0,0.0,0.001662,1.0,0.007887,1.0


## 3. Validation checks

In [8]:
expected_assets = set(SECTOR_ETF_TO_ASSET.values())
actual_assets = set(market_daily_df["asset"].dropna().unique())
missing_assets = expected_assets - actual_assets
assert not missing_assets, f"Missing sector assets: {missing_assets}"

duplicate_rows = market_daily_df.duplicated(["asset", "session_date"]).sum()
assert duplicate_rows == 0, f"Duplicate asset-session rows found: {duplicate_rows}"

# Forward return spot check: today's fwd_1d_return must equal tomorrow's daily_return by asset.
check_df = market_daily_df.sort_values(["asset", "session_date"]).copy()
check_df["next_daily_return"] = check_df.groupby("asset")["daily_return"].shift(-1)
valid_check = check_df[["fwd_1d_return", "next_daily_return"]].dropna()
assert np.allclose(valid_check["fwd_1d_return"], valid_check["next_daily_return"], atol=1e-12), (
    "fwd_1d_return is not shifted correctly"
)

# The overnight gap and the open->close move must compound back to the raw close-to-close price
# change (exactly, up to dividend adjustment which `daily_return` includes and raw prices do not).
recon = market_daily_df.dropna(subset=["overnight_gap", "open_to_close", "prev_close"]).copy()
recon["compounded"] = (1 + recon["overnight_gap"]) * (1 + recon["open_to_close"]) - 1
recon["raw_price_move"] = recon["price"] / recon["prev_close"] - 1
max_gap_error = (recon["compounded"] - recon["raw_price_move"]).abs().max()
assert max_gap_error < 1e-9, f"gap/open-to-close decomposition does not reconcile: {max_gap_error}"

open_missing = market_daily_df["open_price"].isna().mean()
print(f"open_price missing: {open_missing:.2%}")
assert open_missing < 0.02, "more than 2% of sessions are missing an open price"

# How much of the session's move happens before anyone can trade on the news?
share_in_gap = (recon["overnight_gap"].abs().mean()
                / (recon["overnight_gap"].abs().mean() + recon["open_to_close"].abs().mean()))
print(f"share of the average absolute session move that occurs in the overnight gap: {share_in_gap:.1%}")

print("Validation checks passed.")
coverage_summary = market_daily_df.groupby("asset").agg(
    sessions=("session_date", "nunique"),
    start=("session_date", "min"),
    end=("session_date", "max"),
    missing_fwd_1d=("fwd_1d_return", lambda s: s.isna().sum()),
    missing_fwd_5d=("fwd_5d_return", lambda s: s.isna().sum()),
)
display(coverage_summary)

open_price missing: 0.02%
share of the average absolute session move that occurs in the overnight gap: 40.9%
Validation checks passed.


,sessions,start,end,missing_fwd_1d,missing_fwd_5d
asset,,,,,
Communication_Services,1895,2018-06-19,2025-12-31,1.0,5.0
Consumer_Discretionary,2766,2015-01-02,2025-12-31,1.0,5.0
Consumer_Staples,2766,2015-01-02,2025-12-31,1.0,5.0
Energy,2766,2015-01-02,2025-12-31,1.0,5.0
Financials,2766,2015-01-02,2025-12-31,1.0,5.0
Health_Care,2766,2015-01-02,2025-12-31,1.0,5.0
Industrials,2766,2015-01-02,2025-12-31,1.0,5.0
Materials,2766,2015-01-02,2025-12-31,1.0,5.0
Real_Estate,2573,2015-10-08,2025-12-31,1.0,5.0


## 4. Save the gold panel and print a final run summary

`market_daily_df.csv` is engineered (deduplicated, asset-mapped, forward-return features) rather than a raw WRDS export, so it is intended to be committed alongside `news_daily_df.csv` from the RavenPack notebook.

In [9]:
market_daily_df.to_csv(MARKET_DAILY_CSV, index=False)

print("Extraction complete.")
print(f"Bronze/Silver (raw CRSP pull, gitignored): {RAW_PRICES_CSV}  [{len(market_daily_raw):,} rows]")
print(f"Gold (sector panel, committed):            {MARKET_DAILY_CSV}  [{len(market_daily_df):,} rows]")
print(f"Sector ETFs covered: {sorted(market_daily_df['asset'].unique())}")